<a href="https://colab.research.google.com/github/Bertrand94/nitda-blockchain-scholarship/blob/main/curriculum/phase-2a-python/weeks-01-08-teaching/week-03-functions-and-data/02-thursday/exercises/week03_thu_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Functions & First Contact with Data: Reading Real Data with Python
## Phase 2a Python | PORA Academy Cohort 7
### Thursday Exercises

Welcome to the Thursday exercises. Yesterday you wrote functions; today you point
them at a **real file** — the full Olist orders dataset of **99,441 rows** stored in
`olist_orders_dataset.csv`.

**How to work through this notebook:**
- Read each question carefully before writing any code.
- Run the **Data Setup** cell first — every later question depends on it.
- Replace each `# Your code here` line with your own solution.
- Every question lists its **expected output** so you can check yourself.
- Where a question asks for a function, keep the exact function name given — the
  weekly assignment reuses some of them.

There are **no answers in this notebook**. Try each one yourself; struggle is part
of the learning. The final section is your **Weekly Assignment 3**, which you submit
as a separate notebook.

## Data Setup — Run This First

We are **not** using pandas this week. Everything below uses only the standard
library: the `csv` module to parse the file, `os` to build a safe file path, and
`drive` to reach the dataset on your Google Drive.

Run the cell below. When Colab asks for permission to access your Drive, click
through and allow it. After it runs, `orders_filepath` points to the orders CSV and
`load_csv` is ready to use.

In [ ]:
import csv
from google.colab import files
uploaded = files.upload()

Saving olist_orders_dataset.csv to olist_orders_dataset.csv


In [ ]:
import csv
from google.colab import drive
import os

# Mount Google Drive so we can reach the dataset
drive.mount('/content/drive')

# Folder that holds all the Olist CSV files
olist_path = '/content/drive/MyDrive/olist-data'

# Full path to the orders file (os.path.join handles the slash for us)
orders_filepath = os.path.join(olist_path, 'olist_orders_dataset.csv')


def load_csv(filepath, limit=None):
    """Load any CSV file into a list of dicts."""
    rows = []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit and i >= limit:
                break
            rows.append(row)
    return rows


print("Orders file path:", orders_filepath)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Orders file path: /content/drive/MyDrive/olist-data/olist_orders_dataset.csv


## Question 1 — Read the Headers and Count the Rows

Open `orders_filepath` with `open()` and a `csv.DictReader`. Without storing every
row, do two things:
1. Print the column names (`reader.fieldnames`).
2. Loop over the reader and count how many rows the file contains.

**Expected output:**
```
Headers: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Total rows: 99,441
```

Use a `with` block so the file closes itself, and format the count with a thousands
separator (`{row_count:,}`).

In [ ]:
orders_filepath = '/content/olist_orders_dataset.csv' # Overriding orders_filepath to use the locally uploaded file
row_count = 0
with open(orders_filepath, 'r') as f:
    reader = csv.DictReader(f)
    print("Headers:", reader.fieldnames)
    print()
    for row in reader:
              row_count += 1

print(f"Total rows: {row_count:,}")
# Target:
# Total rows: 99,441

Headers: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Total rows: 99,441


## Question 2 — Load a Sample with `load_csv()`

The `load_csv(filepath, limit=None)` function is already defined in the Data Setup
cell. Use it to load just the **first 1,000 orders** as a fast sample, then:
1. Print how many orders you loaded.
2. Print the `order_status` of the very first order.

**Expected output:**
```
Loaded: 1000 orders
Status of first order: delivered
```

Remember each loaded order is a dictionary, so `orders_sample[0]['order_status']`
gives the first order's status.

In [ ]:
def load_orders(filepath, limit=None):
    """
    Load orders CSV into a list of dicts.

    Args:
        filepath: path to CSV file
        limit: max rows to load (None = all rows)
    Returns:
        list of dicts
    """
    orders = []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit and i >= limit:
                break
            orders.append(row)
    return orders



orders_sample = load_orders(orders_filepath, limit=1000)
print(f"Loaded: {len(orders_sample)} orders")
print()
print(f"Status of first order: {orders_sample[0]['order_status']}")



Loaded: 1000 orders

Status of first order: delivered


## Question 3 — Load ALL the Orders

Now load **every** order (no limit) into a list called `orders` and print the total.

**Expected output:**
```
Total orders loaded: 99,441
```

This `orders` list is reused by every question from here on, so make sure it loads
all 99,441 rows before moving on. (Loading the full file may take a few seconds.)

In [ ]:
from collections import Counter
from datetime import datetime


def classify_delivery_days_str(purchase_str, delivery_str):
    """Works directly on the raw string values from the CSV."""
    if not delivery_str:                 # empty string => never delivered
        return "Not Delivered"
    try:
        p = datetime.strptime(purchase_str[:19], "%Y-%m-%d %H:%M:%S")
        d = datetime.strptime(delivery_str[:19], "%Y-%m-%d %H:%M:%S")
        days = (d - p).days
        if days <= 7:
            return "Fast"
        elif days <= 14:
            return "Normal"
        else:
            return "Slow"
    except Exception:
        return "Unknown"


# Load ALL orders this time (limit=None)
orders = load_orders(orders_filepath)
print(f"Total: {len(orders):,}")

# Apply the function to every single order and tally the result
delivery_counts = Counter()
for order in orders:
    speed = classify_delivery_days_str(
        order['order_purchase_timestamp'],
        order['order_delivered_customer_date']
    )
    delivery_counts[speed] += 1

for speed, count in delivery_counts.most_common():
    pct = count / len(orders) * 100
    print(f"  {speed}: {count:,} ({pct:.1f}%)")
# Target:
# Total orders loaded: 99,441

Total: 99,441
  Normal: 36,398 (36.6%)
  Fast: 33,698 (33.9%)
  Slow: 26,380 (26.5%)
  Not Delivered: 2,965 (3.0%)


## Question 4 — Everything from a CSV Is a String

A common surprise: `DictReader` returns **every** value as a string — numbers, dates,
everything. Blank cells come back as the empty string `""`, not `None`.

Using the full `orders` list:
1. Print the `repr()` of the first order's `order_purchase_timestamp` and its type
   name (`type(value).__name__`).
2. Count how many orders have a **blank** `order_delivered_customer_date` (the value
   equals the empty string `""`).

**Expected output:**
```
purchase timestamp value: '2017-10-02 10:56:33'
type: str
Orders with a blank delivery date: 2,965
```

Hint: a generator inside `sum(...)` counts matches in one line —
`sum(1 for o in orders if o['...'] == "")`.

In [ ]:
# Your code here: print the repr() and type name of the first order's purchase timestamp
sample_order = orders[0]
print("purchase timestamp value:", repr(sample_order['order_purchase_timestamp']))
print("type of that value:", type(sample_order['order_purchase_timestamp']).__name__)

# Your code here: count orders whose order_delivered_customer_date == ""
no_delivery = 0  # replace with your counting logic

# Your code here: print  Orders with a blank delivery date: 2,965
no_delivery = sum(1 for o in orders if o['order_delivered_customer_date'] == "")
print(f"Orders with a blank delivery date: {no_delivery:,}")
# Target:
# Orders with a blank delivery date: 2,965

purchase timestamp value: '2017-10-02 10:56:33'
type of that value: str
Orders with a blank delivery date: 2,965


## Question 5 — Write `count_by_status(orders_list)`

Write a function `count_by_status(orders_list)` that loops over the orders and
returns a dictionary mapping each `order_status` to how many times it appears, i.e.
`{status: count}`. Then call it on your `orders` list and print the count for
`"delivered"`.

**Expected output:**
```
delivered: 96,478
```

You may build the dictionary by hand (check `if status in result`) or use
`collections.Counter` — either is fine.

In [ ]:
def count_by_status(orders_list):
    """Return a dict of {order_status: count}."""
    result = {}
    for order in orders_list:
        status = order['order_status']
        if status in result:
            result[status] += 1
        else:
            result[status] = 1
    return result



# Your code here: call count_by_status(orders), then print the "delivered" count
# formatted as  delivered: 96,478
status_counts = count_by_status(orders)
print(f"delivered: {status_counts['delivered']:,}")
# Target:
# delivered: 96,478

delivered: 96,478


## Question 6 — Unique Statuses with a List Comprehension

Using a **list comprehension** (or a set), find all the **distinct** values of
`order_status` across the orders, then report how many there are.

**Expected output (order may vary):**
```
Unique statuses: ['delivered', 'shipped', 'canceled', 'unavailable', 'invoiced', 'processing', 'created', 'approved']
Number of distinct statuses: 8
```

Hint: build a list/set of every order's status, then wrap it in `set(...)` to drop
duplicates, and `len(...)` to count them.

In [ ]:
# Your code here: collect every order_status, reduce to the unique ones
all_statuses = [order['order_status'] for order in orders]

# Your code here: print the list of unique statuses
unique_statuses = sorted(list(set(all_statuses)))
print(f"Unique statuses: {unique_statuses}")
# Your code here: print  Number of distinct statuses: 8
print(f"Number of distinct statuses: {len(unique_statuses)}")
# Target:
# Number of distinct statuses: 8

Unique statuses: ['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']
Number of distinct statuses: 8


## Question 7 — Filter to Cancelled Orders

Build a list containing only the orders whose `order_status` is `"canceled"`
(note the American spelling used in the dataset), then print how many there are.

**Expected output:**
```
Canceled orders: 625
```

Hint: a list comprehension with a condition does this in one line —
`[o for o in orders if o['order_status'] == "canceled"]`.

In [ ]:
# Your code here: filter orders to only "canceled" status
canceled_orders = []  # replace with your filtering logic

# Your code here: print  Canceled orders: 625

canceled_orders = [order for order in orders if order['order_status'] == 'canceled']

print(f"Canceled orders: {len(canceled_orders)}")
# Target:
# Canceled orders: 625

Canceled orders: 625


---
## Thursday Group Exercise: Analyse the Orders Dataset

Work through these four tasks together using the loaded `orders` list (all 99,441
rows). Several of them revisit what you just did above — now connect them into one
short analysis your group can talk through.

```python
# Task 1: Count how many orders have a blank order_delivered_customer_date
#    (Hint: check if the value is an empty string "")
#    Expected: ~2,965 rows have no delivery date

# Task 2: Write a function count_by_status(orders_list) that returns a dict
#    of {status: count} for all statuses in the list
#    Verify: delivered should be 96,478

# Task 3: Using a list comprehension, create a list of all unique order statuses
#    How many distinct statuses are there?

# Task 4: From the orders list, filter to only "canceled" orders
#    How many are there? Expected: 625
```

In [ ]:
import csv
import os
from collections import Counter # Needed for Task 2 and 3

# Define orders_filepath and olist_path (from cell ff5dee73)
olist_path = '/content/drive/MyDrive/olist-data'
orders_filepath = '/content/olist_orders_dataset.csv' # Corrected path to the locally uploaded file

# Define load_orders function (from cell 701bab2e)
def load_orders(filepath, limit=None):
    """
    Load orders CSV into a list of dicts.

    Args:
        filepath: path to CSV file
        limit: max rows to load (None = all rows)
    Returns:
        list of dicts
    """
    orders_list = []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit and i >= limit:
                break
            orders_list.append(row)
    return orders_list

# Now, load the full orders data
# It is recommended to run the cell 2f1af17b to properly load all orders once for efficiency.
orders = load_orders(orders_filepath)

# `orders` is already loaded (all 99,441 rows). Solve each task below.

# Task 1: Count how many orders have a blank order_delivered_customer_date  (expected ~2,965)
blank_delivery_date_count = sum(1 for order in orders if order['order_delivered_customer_date'] == "")
print(f"Orders with a blank delivery date: {blank_delivery_date_count:,}")


# Task 2: Write a function count_by_status(orders_list) -> {status: count}   (delivered == 96,478)
def count_by_status(orders_list):
    return Counter(order['order_status'] for order in orders_list)

status_counts = count_by_status(orders)
print(f"Status counts: {status_counts}")
# Verify delivered count
print(f"Delivered orders (from function): {status_counts['delivered']:,}")


# Task 3: list comprehension of all UNIQUE order statuses, then how many?
all_statuses = [order['order_status'] for order in orders]
unique_statuses = sorted(list(set(all_statuses)))
print(f"Unique statuses: {unique_statuses}")
print(f"Number of distinct statuses: {len(unique_statuses)}")


# Task 4: filter to only "canceled" orders, then count them          (expected 625)
canceled_orders = [order for order in orders if order['order_status'] == 'canceled']
print(f"Canceled orders: {len(canceled_orders)}")

Orders with a blank delivery date: 2,965
Status counts: Counter({'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2})
Delivered orders (from function): 96,478
Unique statuses: ['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']
Number of distinct statuses: 8
Canceled orders: 625


---
## Weekly Assignment 3

Submit `week3_assignment.ipynb`:

1. Write a function `load_csv(filepath, limit=None)` that loads any CSV file into a list of dicts. Test it on `olist_orders_dataset.csv`. Verify row count = **99,441**.

2. Using the loaded orders, write a function `status_report(orders_list)` that:
   - Counts each status
   - Calculates each status as a % of total
   - Returns a list of dicts: `[{"status": "delivered", "count": 96478, "pct": 97.0}, ...]`
   - Print each item as: `"delivered: 96,478 orders (97.0%)"`

3. Write a function `find_orders_by_status(orders_list, status)` that filters and returns only orders matching a given status. Test: `find_orders_by_status(orders, "canceled")` should return a list of **625** orders.

4. **Reflection question** (1 paragraph in a text cell): You just processed 99,441 rows using pure Python. What were the most tedious parts of this work? What would you want a tool to do automatically?

In [ ]:
# Assignment scaffold — copy your finished work into a NEW notebook named
# week3_assignment.ipynb for submission.

# Q1: write load_csv(filepath, limit=None) and verify the row count is 99,441
import csv
from collections import Counter # Required for Q2

def load_csv(filepath, limit=None):
    """Load any CSV file into a list of dicts."""
    rows = []
    with open(filepath, 'r') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit and i >= limit:
                break
            rows.append(row)
    return rows

orders_filepath = '/content/olist_orders_dataset.csv' # Using the corrected local path
all_orders = load_csv(orders_filepath)
print(f"Q1 - Total rows loaded: {len(all_orders):,}")


# Q2: write status_report(orders_list) returning a list of dicts and printing each line
def status_report(orders_list):
    """
    Counts each order status, calculates percentages, and returns a list of dicts.
    """
    total_orders = len(orders_list)
    status_counts = Counter(order['order_status'] for order in orders_list)
    report = []
    for status, count in status_counts.items():
        pct = (count / total_orders) * 100
        report.append({"status": status, "count": count, "pct": round(pct, 1)})

    report.sort(key=lambda x: x['count'], reverse=True)
    return report

print("\nQ2 - Status Report:")
report_data = status_report(all_orders)
for item in report_data:
    print(f"{item['status']}: {item['count']:,} orders ({item['pct']:.1f}%)")


# Q3: write find_orders_by_status(orders_list, status); test "canceled" -> 625 orders
def find_orders_by_status(orders_list, status):
    """
    Filters and returns orders matching a given status.
    """
    return [order for order in orders_list if order['order_status'] == status]

print("\nQ3 - Testing find_orders_by_status:")
canceled_orders_assignment = find_orders_by_status(all_orders, "canceled")
print(f"Number of canceled orders found: {len(canceled_orders_assignment)}")

Q1 - Total rows loaded: 99,441

Q2 - Status Report:
delivered: 96,478 orders (97.0%)
shipped: 1,107 orders (1.1%)
canceled: 625 orders (0.6%)
unavailable: 609 orders (0.6%)
invoiced: 314 orders (0.3%)
processing: 301 orders (0.3%)
created: 5 orders (0.0%)
approved: 2 orders (0.0%)

Q3 - Testing find_orders_by_status:
Number of canceled orders found: 625


### Q4 - Reflection Question

Processing 99,441 rows using pure Python, the most tedious part was definitely the manual parsing and data type conversion. Since `csv.DictReader` returns every value as a string, handling dates and potentially numbers required explicit `datetime.strptime` calls and error handling for empty strings or malformed data. Additionally, aggregating and calculating percentages involved writing custom loops and `Counter` logic. A tool like Pandas would automate much of this, offering built-in functions for reading CSVs with schema inference, vectorized operations for calculations, and robust methods for handling missing values and data type conversions, significantly reducing boilerplate code and improving readability.